In [1]:
# Kaggle: turn Internet ON (Notebook options > Internet) before running this.
# Versions are pinned to the ones Unsloth's own Gemma3 (4B) notebook uses -- unsloth
# breaks easily against a mismatched transformers. Restart the session afterwards if
# transformers was already imported in this kernel.
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/22

In [2]:
import pandas as pd
from tabulate import tabulate

df = pd.read_json('/kaggle/input/datasets/hieu2k5/vlsp2025-vinumqa/test.json')
df.sample(n=5)

,pre_text,table,post_text,id,qa
403,[đánh giá quý 1: phương pháp quản trị thận trọ...,"[[Năm tài chính (31/12), 12/16, 12/17, 12/18, ...",[.],masvn/2020/2020053-200427_VCB_1Q20_review_vn/p...,{'question': 'Tỷ lệ P/E dự báo cho năm 2022 gi...
35,[thuyết minh báo cáo tài chính hợp nhất (tiếp ...,"[[, 2006, 2005 đã điều chỉnh lại (1), 2004 đã ...","[(1) xem ghi chú 2, “điều chỉnh lại báo cáo tà...",AAPL/2006/page_100.pdf-4,{'question': 'Tỷ lệ thuế hiệu quả thấp nhất tr...
127,"[republic services , inc. thuyết minh báo cáo ...","[[, ngày 31 tháng 12 năm 2018 phân bổ tài sản ...",[việc phân bổ tài sản được xem xét và tái cân ...,RSG/2018/page_135.pdf-1,{'question': 'Dựa trên mục tiêu ngày 31 tháng ...
376,[công ty có các khoản lỗ vốn được chuyển sang ...,"[[số dư tại ngày 1 tháng 1 năm 2011, $ 118314]...",[số dư nợ phải trả bao gồm các khoản được phản...,AWK/2012/page_117.pdf-3,{'question': 'Tổng thay đổi trong các lợi ích ...
364,"[sử dụng phương pháp p/b và rnav để định giá, ...","[[Năm tài chính (31/12), 2016, 2017, 2018, 201...",[.],masvn/2020/2020193-VN_IndustrialRealEstate_Upd...,{'question': 'Tổng tỷ lệ sở hữu của Công ty cổ...


In [3]:
len(df)

497

In [4]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
df["input_question"] = df.apply(lambda x: processing_input_question(x), axis=1)
df["program_processed"] = df.apply(lambda x: processing_program_content(x), axis=1)
df["answer_processed"] = df.apply(lambda x: processing_answer_content(x), axis=1)
df.sample(n=5)

,pre_text,table,post_text,id,qa,pre_text_processed,post_text_processed,table_processed,table_raw,input_question,program_processed,answer_processed
493,"[định giá và khuyến nghị:, chúng tôi khuyến ng...","[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",[.],masvn/2020/2020145-VN_POW_Update_BUY_MAS_20201...,{'question': 'Doanh thu trung bình từ năm tài ...,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,.,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",Doanh thu trung bình từ năm tài chính 2017 đến...,"add(29710, 32662), add(#0, 35374), divide(#1, 3)",32582.0
115,"[định giá và khuyến nghị:, chúng tôi khuyến ng...","[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",[.],masvn/2020/2020145-VN_POW_Update_BUY_MAS_20201...,{'question': 'Lợi nhuận ròng năm tài chính 202...,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,.,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",Lợi nhuận ròng năm tài chính 2021 (FY21F) được...,"subtract(2371, 1945), divide(#0, 1945)",0.21902
116,"[định giá và khuyến nghị, chúng tôi duy trì kh...","[[FY (tỷ đồng), FY18, FY19, FY20, FY21, FY22],...",[.],masvn/2020/2020163-VN_DHG_Update_BUY_MAS_20201...,{'question': 'Tỷ lệ thay đổi phần trăm của Lợi...,định giá và khuyến nghị\nchúng tôi duy trì khu...,.,| FY (tỷ đồng) | FY18 | FY19 ...,"[[FY (tỷ đồng), FY18, FY19, FY20, FY21, FY22],...",Tỷ lệ thay đổi phần trăm của Lợi nhuận trên mỗ...,"subtract(5738, 5687), divide(#0, 5687)",0.00897
108,"[đối với trái phiếu 4,25% (4,25%) đáo hạn vào ...","[[năm, số tiền], [2015, $ 126], [2016, 111], [...",[chi phí thuê và chi phí thiết bị văn phòng nh...,BLK/2014/page_120.pdf-1,{'question': 'Bao nhiêu phần trăm các cam kết ...,"đối với trái phiếu 4,25% (4,25%) đáo hạn vào n...",chi phí thuê và chi phí thiết bị văn phòng nhấ...,| năm | số tiền |\n|-----------|------...,"[[năm, số tiền], [2015, $ 126], [2016, 111], [...",Bao nhiêu phần trăm các cam kết đáo hạn sau nă...,"divide(613, 1178)",0.52037
319,[tóm tắt các thay đổi theo fin 48 trong năm tà...,[[số dư đầu kỳ tính đến ngày 1 tháng 12 năm 20...,[tổng nợ phải trả cho các lợi ích thuế chưa đư...,ADBE/2008/page_89.pdf-1,{'question': 'Thay đổi ròng trong tổng nợ phải...,tóm tắt các thay đổi theo fin 48 trong năm tài...,tổng nợ phải trả cho các lợi ích thuế chưa đượ...,| số dư đầu kỳ tính đến ngày 1 tháng 12 năm 20...,[[số dư đầu kỳ tính đến ngày 1 tháng 12 năm 20...,Thay đổi ròng trong tổng nợ phải trả cho các l...,"subtract(139549, 201808)",-62259.0


In [5]:
df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
df["generated_program"] = ""
df["calculated_program"] = ""
df

,pre_text,table,table_raw,post_text,question,program,answer,generated_program,calculated_program
0,thuyết minh báo cáo tài chính hợp nhất ( tiếp ...,| các thành phần của ảnh hưởng lũy kế của việc...,[[các thành phần của ảnh hưởng lũy kế của việc...,.,Sự thay đổi trong thu nhập ròng từ hiệu ứng tí...,"add(30, 1)",31.0,,
1,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,"Theo dự phóng, doanh thu và lợi nhuận ròng quý...","subtract(9829, 642)",9187.0,,
2,"sử dụng phương pháp p/b và rnav để định giá, c...",| Năm tài chính (31/12) | 2016 | 2017 | ...,"[[Năm tài chính (31/12), 2016, 2017, 2018, 201...",.,IDC có bao nhiêu ha quỹ đất sẵn sàng cho thuê ...,"add(495, 398)",893.0,,
3,27/10/13 26/10/14 25/10/15 30/10/16 29/10/17 2...,| | 27/10/2013 |...,"[[, 27/10/2013, 26/10/2014, 25/10/2015, 30/10/...",.,Tỷ suất lợi nhuận trên đầu tư (ROI) của Applie...,"subtract(96.67, 100), divide(#0, 100)",-0.0333,,
4,thông tin tài chính bổ sung hiệu suất cổ phiếu...,| | 12/26/08 | ...,"[[, 12/26/08, 12/31/09, 12/31/10, 12/31/11, 12...",218 báo cáo thường niên năm 2013 của goldman s...,tỷ lệ lợi nhuận tích lũy tổng cộng theo phần t...,"subtract(248.36, 100), divide(#0, 100)",1.4836,,
...,...,...,...,...,...,...,...,...,...
492,thuyết minh báo cáo tài chính hợp nhất năm 201...,| ...,"[[, 2008, 2007], [Số dư đầu kỳ, $ 134.8, $ 266...",trong tổng số lợi ích thuế chưa được ghi nhận ...,Tỷ lệ phần trăm lợi ích thuế chưa được công nh...,"divide(131.8, 148.8)",0.88575,,
493,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Doanh thu trung bình từ năm tài chính 2017 đến...,"add(29710, 32662), add(#0, 35374), divide(#1, 3)",32582.0,,
494,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Tính phần trăm thay đổi EPS từ năm 2020 đến 2021.,"subtract(962, 788), divide(#0, 788)",0.22081,,
495,"trong quá trình kinh doanh thông thường, dựa t...",| ( đơn vị: nghìn ) | diện tích ròng chưa ph...,"[[( đơn vị: nghìn ), diện tích ròng chưa phát ...",( a ) một giếng khoan thăm dò được lên kế hoạc...,Tỷ lệ phần trăm diện tích đất chưa phát triển ...,"divide(145, 586)",0.24744,,


In [6]:
# unsloth must be imported before transformers so its patches take effect.
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

import gc
import torch
import pandas as pd
from tqdm import tqdm

# No huggingface_hub login here: unsloth/gemma-3-4b-it is an ungated mirror of
# google/gemma-3-4b-it, so neither an HF_TOKEN nor accepting the Gemma licence is
# needed. (Kaggle does not expose Add-ons > Secrets as env vars anyway.)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
model_name = "unsloth/gemma-3-4b-it"

# Unsloth's reference notebook uses max_seq_length = 2048, which is too short here:
# measured over datasets/ViNumQA/test.json, the longest 0-shot prompt is ~3.7k tokens
# and ~21% of the 497 items exceed 2048, so those prompts would be silently truncated
# and score 0 for reasons that have nothing to do with the model. 8192 covers the
# longest prompt plus the 512 generated tokens.
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,   # keep 16-bit weights so the score is comparable with the Qwen3-4B run
    load_in_8bit = False,
    full_finetuning = False,
)

# Gemma 3's own template: it folds the system turn into the first user turn and
# renders the assistant role as "model".
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

==((====))==  Unsloth 2026.7.5: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}
 
[TABLE]
{table}
 
[TEXT AFTER TABLE]
{post_text}
 
### QUESTION:
{question}
 
### PROGRAM:"""


In [9]:
# Gemma 3's sampling defaults, passed explicitly to generate() as Unsloth's notebook does.
GEN_KWARGS = dict(
    max_new_tokens = 512,
    do_sample = True,
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64,
)

n_too_long = 0

for df_index, values in tqdm(df.iterrows(), total=len(df), desc="Generating program..."):
    pre_text = values["pre_text"]
    table = values["table"]
    post_text = values["post_text"]
    question = values["question"]

    USER_MESSAGE = USER_MESSAGE_FRAME.format(
        pre_text=pre_text,
        table=table,
        post_text=post_text,
        question=question
    )

    # Gemma 3 takes the content of each turn as a list of typed parts. There is no
    # thinking mode, so unlike Qwen3 there is no enable_thinking flag and no </think>
    # marker to slice the answer out of.
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_MESSAGE}]},
        {"role": "user", "content": [{"type": "text", "text": USER_MESSAGE}]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_tensors = "pt",
        return_dict = True,
    ).to("cuda")

    input_len = inputs["input_ids"].shape[-1]
    if input_len + GEN_KWARGS["max_new_tokens"] > MAX_SEQ_LENGTH:
        n_too_long += 1

    with torch.inference_mode():
        outputs = model.generate(**inputs, **GEN_KWARGS)

    # Decode only the newly generated tokens, i.e. everything past the prompt.
    output = tokenizer.batch_decode(
        outputs[:, input_len:], skip_special_tokens=True
    )[0].strip()

    df.at[df_index, "generated_program"] = output

    # print(f"TEST SAMPLE {df_index}:\n\nPREDICTION:\n{output}\n\nGROUND_TRUTH:\n{values['program']}")
    # print("="*100)

    gc.collect()
    torch.cuda.empty_cache()

if n_too_long:
    print(f"WARNING: {n_too_long} prompts did not fit in MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} "
          f"together with max_new_tokens -- raise MAX_SEQ_LENGTH and re-run.")

Generating program...: 100%|██████████| 497/497 [2:28:48<00:00, 17.97s/it]


In [10]:
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented:

  * Program Accuracy is *symbolic* equivalence, via sympy, between the gold and
    predicted expressions -- not a string or structural match. A prediction may
    reorder or restructure the arithmetic, but it may only use literals that
    appear in the gold program, so it cannot invent constants such as the `100`
    of a percentage rescaling.
  * Execution Accuracy compares the executed result to `exe_ans` exactly, after
    rounding to 5 decimals. No tolerance.
  * `greater` yields the strings "yes"/"no", matching how the dataset stores
    those answers.
  * Every step takes exactly two arguments. Verified against the data: all 663
    steps across the gold programs are binary, and `table_*` always takes a row
    label plus `none` (454 occurrences) rather than a list of values (2).
  * FinQA's `const_` tokens are still understood.

Five corrections are applied, each because the unmodified script cannot
reproduce ViNumQA's own gold, not because the protocol was thought wrong:

1. Tokenisation of bracketed row labels. `program_tokenization` splits on every
   bracket, so `table_min(ROE (%), none)` shatters into six tokens and fails the
   four-tokens-per-step structure check. 35 of the 497 test programs name a row
   whose label contains brackets -- `ROE (%)`, `EPS (VND)`, `P/E (x)` -- and all
   35 were unscoreable. Tokenisation is now bracket-depth aware.

2. Accounting negatives. Tables write negative amounts as `(3344)`. The original
   `process_row` takes the text before the first bracket, leaving an empty
   string, so the cell fails to parse. The dataset's own `exe_ans` was computed
   with those values -- e.g. `table_min(LN hoạt động (tỷ đồng), none)` expects
   -3344 from a row holding `(3344)`. The `-1046 ( 1046 )` form the original
   handled correctly is unchanged.

3. Unparseable cells no longer void the whole row. Measured over the 393 gold
   `table_*(<row>, none)` programs in train, skipping such cells reproduces
   `exe_ans` for 386 against 381 when the row is voided, so skipping is what the
   dataset was built with.

4. `exe_ans` is stored as a string here ("31.0") where FinQA stores a number, so
   the comparison `exe_res == gold_res` was never true. It is coerced, leaving
   the "yes"/"no" answers alone.

5. The `assert exe_res == gold_res` inside the program-accuracy branch is
   dropped. It is a debug check, and a single rounding disagreement aborts the
   whole evaluation.

`evaluate_result_official` runs the unmodified protocol for comparison, so the
cost of each correction can be seen rather than assumed.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


# ------------------------------------------------------------------ numbers --
def str_to_num(text: str) -> Union[float, str]:
    """FinQA's literal parser, unchanged: returns "n/a" rather than raising."""
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            # Multiples are written "14.3x" in these tables. Unlike "%", the
            # suffix carries no scaling -- the gold answer for such a row is the
            # plain multiple.
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    """Parse one table cell.

    Adds the `(3344)` form to what the original handled; `$ -1046 ( 1046 )`
    still resolves through the original's "text before the first bracket" rule.
    """
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    """Numeric values of a table row, or "n/a" if the row cannot be reduced.

    A cell that merely marks a missing period ("-", "NA", an em dash) is
    skipped: rows in this dataset routinely lack a year or two, and voiding the
    whole row over one gap loses reductions the gold answers depend on. A cell
    with real but unreadable content still voids the row, so genuine parse
    failures are not silently averaged away.
    """
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


# -------------------------------------------------------------- tokenisation --
def program_tokenization(original_program: str) -> List[str]:
    """Tokenise into ['op(', arg1, arg2, ')', ..., 'EOF'].

    Bracket-depth aware, so a row label like `ROE (%)` stays one token. The
    original split on every bracket, which shattered such labels and broke the
    four-tokens-per-step structure the rest of the protocol relies on.

    Raises ValueError if trailing, non-whitespace text remains once no further
    step can be parsed (e.g. a step missing its closing paren, which happens
    both in a handful of gold programs and -- more importantly -- in model
    generations cut off by a max_new_tokens limit). An earlier version of this
    tokenizer silently stopped and returned only the steps parsed so far,
    which let a truncated program like "subtract(100, 50), divide(#0, 5"
    (missing text and closing paren) score as a valid, complete one-step
    program instead of being rejected -- a false positive for exactly the kind
    of generation failure this evaluator needs to catch.
    """
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching ')' found) in program: '{original_program}'"
            )

        program.append(m.group(1) + "(")
        # Split arguments on depth-0 commas so brackets inside a label survive.
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: '{text[pos:]}' (from: '{original_program}')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    """Recover a program string from raw model output.

    Bracket matched, so an outer call is never silently discarded: the earlier
    regex could only match a bracket-free call, so `multiply(divide(a, b), 100)`
    was reduced to its inner `divide(a, b)` and a percentage rescaling scored as
    if it were the gold answer.
    """
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text



def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    """Group a tokenised program into (op, arg1, arg2) triples.

    The original walked the token list by joining it and splitting on ")", which
    silently mis-splits any argument containing a bracket -- exactly the row
    labels this dataset uses, e.g. `EPS (VND)`. Grouping the tokens directly is
    equivalent for well-formed programs and correct for those.
    """
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps

# ---------------------------------------------------------------- execution --
def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    """Execute a tokenised program. Returns (invalid_flag, result)."""
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


# ------------------------------------------------------------------ program --
def equal_program(program1: List[str], program2: List[str]) -> bool:
    """Symbolic equivalence of gold (program1) and prediction (program2).

    Same protocol as the official implementation -- literals become symbols,
    table steps become opaque variables, and the two expressions are compared
    after `simplify`, so a differently-arranged but algebraically identical
    program still counts. A prediction may only use symbols that appear in gold,
    which is what stops it from introducing a constant of its own (the `100` of
    a percentage rescaling, say). Only the step-splitting differs: it groups
    tokens rather than splitting a joined string on ")".
    """
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


# ------------------------------------------------------------------ metrics --
def _coerce_answer(value):
    """ViNumQA stores exe_ans as a string; "yes"/"no" stay as they are."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    """(program_accuracy, execution_accuracy) for a single item.

    A generated_program that fails to tokenize (e.g. cut off mid-generation,
    missing a closing paren) scores (0.0, 0.0) rather than raising -- this is
    expected input from a real model, not a bug to surface as an exception.
    gold_program is assumed well-formed and is not caught the same way, so a
    malformed *gold* label still raises loudly instead of silently scoring 0.
    """
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


def evaluate_dataframe(df, generated_col: str = "generated_program",
                       gold_program_col: str = "program",
                       gold_answer_col: str = "answer",
                       table_col: str = "table_raw",
                       extract_first: bool = True):
    """Score a DataFrame, returning (df + per-row scores, summary).

    `table_col` must hold the raw table (list of rows); without it, programs
    naming a table row cannot execute and score 0 on EA.
    """
    df = df.copy()
    pa_scores, ea_scores = [], []

    for _, row in df.iterrows():
        table = row[table_col] if table_col in df.columns else None
        pa, ea = score_one(row[generated_col], row[gold_program_col],
                           row[gold_answer_col], table, extract_first)
        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    return df, {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }

In [11]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",   # cột bạn đang ghi output model vào
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}

{'program_accuracy': 0.018108651911468814, 'execution_accuracy': 0.04627766599597585}
